# CoverageEvaluator usage

Coverage measures how much materially important information from `context` is represented in `output`. Higher is better. It requires `context + output`; other case fields are ignored. One case makes one judge call.

In [ ]:
from idp_eval import (
    CoverageEvaluator,
    EvaluationCase,
    EvaluationFramework,
    create_azure_judge,
)
from idp_eval.judges import AzureJudgeConfig

## Configure a judge

Use placeholders only. Production applications normally populate this config from their settings/secrets layer. Evaluators are backend-independent; `create_gateway_judge(config=gateway_config)` can be used instead. See the setup guide and backend latency notebook for backend configuration.

In [ ]:
azure_config = AzureJudgeConfig(
    model="your-azure-deployment",
    azure_endpoint="https://your-resource.openai.azure.com",
    tenant_id="your-tenant-id",
    client_id="your-client-id",
    client_secret="your-client-secret",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)
judge = create_azure_judge(config=azure_config)

Structured values are supported generically: dictionary keys become readable labels, lists become bullets, and nested structures render recursively. A list in `case.output` remains one structured output and one case.

In [ ]:
case = EvaluationCase(
    context={
        "requirements": [
            {"name": "Audit logging", "mandatory": True},
            {"name": "Regional hosting", "regions": ["US", "EU"]},
        ]
    },
    output={
        "summary": "Use audit logging and regional hosting.",
        "regions": ["US", "EU"],
    },
)
evaluator = CoverageEvaluator(judge, verbose=True)
framework = EvaluationFramework(
    judge=judge,
    evaluators=[evaluator],
)
result = framework.evaluate(case)["coverage"]
{
    "score": result.score,
    "label": result.label,
    "explanation": result.explanation,
    "details": result.details,
}

## Optional async equivalent

Jupyter supports top-level `await`; this is an alternative execution example and reuses the same setup.

In [ ]:
async_result = await framework.a_evaluate(case)
async_result["coverage"]

In [ ]:
judge.close()